# ログエラー検知ハンズオン (Databricks Free Edition)

Oracle アラートログ × Java アプリログから **エラーを検知する** ミニ基盤を、
Medallion アーキテクチャ (Bronze → Silver → Gold) で構築します。

**ストーリー:** 2026-06-20 10:02 頃に Oracle 側でデッドロック (ORA-00060) → 接続枯渇
(ORA-12516 / ORA-12519) が発生し、ほぼ同時刻に Java アプリ側で `SQLException` /
コネクションプールのタイムアウトが急増するインシデントが起きています。
Gold 層で時間窓集計すると、この **DB 障害とアプリ障害の相関** が浮かび上がります。

| 層 | 役割 | このノートブックでの内容 |
|----|------|------------------------|
| **Bronze** | 生ログをそのまま保管 | ログファイルを1行=1レコードで Delta テーブル化 |
| **Silver** | 構造化・クレンジング | 正規表現でタイムスタンプ/レベル/エラーコードを抽出、スタックトレースを結合、`.trc` をファイル単位で構造化 |
| **Gold** | 集計・検知 | 時間窓ごとのエラー件数、しきい値超過アラート、DB×アプリ相関 |

> Free Edition はサーバーレス専用・Unity Catalog 前提です。クラスター作成は不要で、
> セル実行時に自動でサーバーレスコンピュートにアタッチされます。


## Step 0. カタログ・スキーマ・ボリュームを作成

データの置き場所 (Unity Catalog) を用意します。`CREATE CATALOG` が権限エラーになる場合は、
既定の `workspace` カタログを使うよう、以下の `CATALOG` 変数だけ書き換えてください
(その場合 `CREATE CATALOG` 行はコメントアウト)。


In [ ]:
%sql
-- 失敗する場合は次行をコメントアウトし、以降の loghandson を workspace に読み替え
CREATE CATALOG IF NOT EXISTS loghandson;
CREATE SCHEMA  IF NOT EXISTS loghandson.logs;
-- 生ログファイルを置くための Volume (ストレージ領域)
CREATE VOLUME  IF NOT EXISTS loghandson.logs.raw;

### ファイルのアップロード

左サイドバー **Catalog** → `loghandson` → `logs` → Volumes → `raw` を開き、
右上の **Upload to this volume** から次のファイルをアップロードしてください。

- `alert_ORCL.log` (Oracle アラートログ)
- `app.log` (Java アプリログ)
- `*.trc` (Oracle トレースファイル / 任意の枚数)

トレースファイルは `raw` 直下でも、`raw/trace/` のようなサブフォルダでも構いません
(後段の取り込みは `raw` 配下を再帰的に探索します)。アップロード後のパスは
`/Volumes/loghandson/logs/raw/...` になります。

> **補足:** アラートログの `ORA-00060` 行には `More info in file ...orcl_ora_XXXX.trc` と
> トレースファイル名が書かれています。基本は同じ階層に実体がありますが、無いものもあるため、
> Step 3 で「参照あり・実体なし」を検出できるようにしています。


In [ ]:
# パラメータ。CATALOG を変えたい場合はここだけ修正
CATALOG = "loghandson"
SCHEMA  = "logs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# アップロード済みファイルの確認
import os
print(os.listdir(VOLUME_PATH))

## Step 1. Bronze 層 — 生ログを1行=1レコードで取り込む

Bronze は「**加工せず、来たものをそのまま貯める**」層です。
ログの行順を後段のスタックトレース結合に使うため、`line_no`(行番号) を付けて取り込みます。

> Free Edition はサーバーレスのため DBFS が制限されています。Volume はローカルパスとして
> マウントされるので、ここでは Python で開いて `line_no` 付きの DataFrame にします
> (ログは小さいので問題ありません)。


In [ ]:
def load_raw(filename: str):
    """Volume 上のテキストログを (line_no, raw) の DataFrame として読み込む。"""
    path = f"{VOLUME_PATH}/{filename}"
    with open(path, "r", encoding="utf-8") as f:
        rows = [(i, line.rstrip("\n")) for i, line in enumerate(f)]
    return spark.createDataFrame(rows, "line_no INT, raw STRING")

bronze_oracle = load_raw("alert_ORCL.log")
bronze_app    = load_raw("app.log")

bronze_oracle.write.mode("overwrite").saveAsTable("bronze_oracle_raw")
bronze_app.write.mode("overwrite").saveAsTable("bronze_app_raw")

print("oracle rows:", bronze_oracle.count(), " app rows:", bronze_app.count())
display(bronze_app.orderBy("line_no").limit(8))

## Step 2. Silver 層① — Oracle アラートログの構造化

Oracle アラートログは **「タイムスタンプ行」+「本文行」のペア**で1イベントです。

```
2026-06-20T10:02:14.000000+09:00
ORA-00060: Deadlock detected. More info in file ...
```

行番号でグルーピングし、ヘッダ行とその後続行を1イベントに束ねてから、
タイムスタンプ・`ORA-xxxxx` コード・重大度を抽出します。


In [ ]:
from pyspark.sql import functions as F, Window

# ヘッダ行 = ISO タイムスタンプで始まる行
TS_HEADER = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"

w = Window.orderBy("line_no")
df = (spark.table("bronze_oracle_raw")
        .withColumn("is_header", F.col("raw").rlike(TS_HEADER))
        # ヘッダ行の line_no を後続行に前方補完 → イベント ID
        .withColumn("event_id",
                    F.last(F.when(F.col("is_header"), F.col("line_no")), ignorenulls=True).over(w)))

# イベント単位に全行を結合
events = (df.groupBy("event_id")
            .agg(F.concat_ws("\n", F.collect_list("raw")).alias("full_text")))

silver_oracle = (events
    .withColumn("event_time",
        F.to_timestamp(F.regexp_extract("full_text",
            r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+)", 1),
            "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"))
    .withColumn("ora_code", F.regexp_extract("full_text", r"(ORA-\d{5})", 1))
    .withColumn("message",
        F.regexp_replace("full_text", r"^.*?\n", ""))   # 1行目(ts)を除いた本文
    .withColumn("severity",
        F.when(F.col("ora_code") != "", F.lit("ERROR")).otherwise(F.lit("INFO")))
    # アラート本文に書かれたトレースファイル名 (例 orcl_ora_10233.trc) を結合キーとして抽出
    .withColumn("trace_file", F.regexp_extract("full_text", r"([A-Za-z0-9_]+\.trc)", 1))
    .select("event_time", "severity", "ora_code", "trace_file", "message")
    .filter(F.col("event_time").isNotNull()))

silver_oracle.write.mode("overwrite").saveAsTable("silver_oracle")
display(silver_oracle.filter("severity = 'ERROR'").orderBy("event_time"))

## Step 3. トレースファイル (.trc) の取り込み — Bronze + Silver

`ORA-00060` のアラートは詳細を別ファイル(`*.trc`)に書き出します。トレースファイルは
**1ファイルがまるごと1イベント**(複数行のダンプ)なので、行単位ではなく
**ファイル単位**で取り込み、ヘッダのメタ情報(発生時刻・インスタンス・PID・モジュール・
デッドロックの SQL など)を抽出して `silver_trace` にします。

最後に `silver_oracle.trace_file` と突き合わせて、**「アラートが参照しているのに実体が
無い trc」** を検出します。


In [ ]:
import os

# Bronze: raw 配下を再帰的に探索し、.trc を「ファイル単位」で取り込む
def find_trace_files(root):
    hits = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.endswith(".trc"):
                hits.append(os.path.join(dirpath, fn))
    return hits

trace_paths = find_trace_files(VOLUME_PATH)
rows = []
for p in trace_paths:
    with open(p, "r", encoding="utf-8", errors="replace") as f:
        rows.append((os.path.basename(p), p, f.read()))

# 0件でもスキーマ付き空テーブルを作れるようにする
bronze_trace = spark.createDataFrame(rows, "trace_file STRING, source_path STRING, full_text STRING")
bronze_trace.write.mode("overwrite").saveAsTable("bronze_trace_raw")
print("trace files loaded:", bronze_trace.count())
display(bronze_trace.select("trace_file", "source_path"))

In [ ]:
from pyspark.sql import functions as F

def rx(pattern, grp=1):
    return F.regexp_extract("full_text", pattern, grp)

silver_trace = (spark.table("bronze_trace_raw")
    # ヘッダの "*** <timestamp>" から発生時刻
    .withColumn("trace_time",
        F.to_timestamp(rx(r"\*\*\* (\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+)"),
                       "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"))
    .withColumn("db_version",   rx(r"Version (\d+\.\d+\.\d+\.\d+\.\d+)"))
    .withColumn("instance_name", rx(r"Instance name:\s*(\S+)"))
    .withColumn("node_name",     rx(r"Node name:\s*(\S+)"))
    .withColumn("unix_pid",      rx(r"Unix process pid:\s*(\d+)"))
    .withColumn("session_id",    rx(r"SESSION ID:\(([^)]*)\)"))
    .withColumn("module_name",   rx(r"MODULE NAME:\(([^)]*)\)"))
    .withColumn("service_name",  rx(r"SERVICE NAME:\(([^)]*)\)"))
    .withColumn("action_name",   rx(r"ACTION NAME:\(([^)]*)\)"))
    # ダンプ種別と ORA コード
    .withColumn("dump_kind",
        F.when(F.col("full_text").rlike("DEADLOCK DETECTED"), F.lit("DEADLOCK"))
         .when(F.col("full_text").rlike("ORA-04031"), F.lit("SHARED_POOL"))
         .otherwise(F.lit("OTHER")))
    .withColumn("ora_code",
        F.when(F.col("full_text").rlike("DEADLOCK DETECTED"), F.lit("ORA-00060"))
         .otherwise(rx(r"(ORA-\d{5})")))
    # "Current SQL Statement for this session" 直後の1行(代表 SQL)
    .withColumn("current_sql",
        rx(r"----- Current SQL Statement for this session \([^)]*\) -----\n(.*)"))
    .select("trace_file", "trace_time", "ora_code", "dump_kind",
            "instance_name", "node_name", "unix_pid", "session_id",
            "module_name", "service_name", "action_name", "current_sql", "source_path"))

silver_trace.write.mode("overwrite").saveAsTable("silver_trace")
display(silver_trace.orderBy("trace_time"))

### 突合検証 — アラートが参照する trc の有無

`silver_oracle`(アラート側)を起点に `silver_trace` を LEFT JOIN し、参照されている
トレースファイルが実在するか(`matched`)、参照のみで実体が無いか(`⚠️ trace missing`)を
一覧します。サンプルでは `orcl_ora_10258.trc` をあえて未配置にしているので、1件が
missing として出ます。


In [ ]:
%sql
SELECT
  o.event_time,
  o.ora_code,
  o.trace_file,
  CASE WHEN o.trace_file = '' THEN '(no trace ref)'
       WHEN t.trace_file IS NULL THEN '⚠️ trace missing'
       ELSE 'matched' END AS trace_status,
  t.dump_kind,
  t.current_sql
FROM silver_oracle o
LEFT JOIN silver_trace t ON o.trace_file = t.trace_file
WHERE o.ora_code = 'ORA-00060'
ORDER BY o.event_time;

## Step 4. Silver 層③ — Java アプリログの構造化 (マルチライン対応)

Java ログは1イベントが複数行になります (メッセージ行 + スタックトレース)。

```
2026-06-20 10:02:14,123 ERROR [http-nio-8080-exec-7] c.e.service.OrderService - Failed to place order...
org.springframework.dao.CannotAcquireLockException: ... [ORA-00060: deadlock detected]
	at ...
Caused by: java.sql.SQLException: ORA-00060: ...
```

ヘッダ行 (タイムスタンプ始まり) を起点にスタックトレースを結合し、
**スタックトレース内の `ORA-xxxxx` も抽出**します。これが後で DB ログとの相関キーになります。


In [ ]:
# ヘッダ行 = "yyyy-MM-dd HH:mm:ss,SSS LEVEL [thread] logger - msg"
APP_HEADER = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}\s"

w = Window.orderBy("line_no")
df = (spark.table("bronze_app_raw")
        .withColumn("is_header", F.col("raw").rlike(APP_HEADER))
        .withColumn("event_id",
                    F.last(F.when(F.col("is_header"), F.col("line_no")), ignorenulls=True).over(w)))

# event_id ごとに、ヘッダ行(=最小 line_no)と全文を作る
hdr = df.filter("is_header").select(F.col("event_id"), F.col("raw").alias("header"))
full = (df.groupBy("event_id")
          .agg(F.concat_ws("\n", F.collect_list("raw")).alias("full_text")))
joined = hdr.join(full, "event_id")

APP_RE = r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s+(\w+)\s+\[([^\]]+)\]\s+(\S+)\s+-\s+(.*)$"

silver_app = (joined
    .withColumn("event_time",
        F.to_timestamp(F.regexp_extract("header", APP_RE, 1), "yyyy-MM-dd HH:mm:ss,SSS"))
    .withColumn("level",  F.regexp_extract("header", APP_RE, 2))
    .withColumn("thread", F.regexp_extract("header", APP_RE, 3))
    .withColumn("logger", F.regexp_extract("header", APP_RE, 4))
    .withColumn("message", F.regexp_extract("header", APP_RE, 5))
    # 例外クラス(スタック1行目)と、スタック全体に含まれる ORA コードを抽出
    .withColumn("exception_class",
        F.regexp_extract("full_text", r"\n([a-zA-Z0-9_.]+(?:Exception|Error))", 1))
    .withColumn("ora_code", F.regexp_extract("full_text", r"(ORA-\d{5})", 1))
    .select("event_time", "level", "thread", "logger", "message",
            "exception_class", "ora_code")
    .filter(F.col("event_time").isNotNull()))

silver_app.write.mode("overwrite").saveAsTable("silver_app")
display(silver_app.filter("level = 'ERROR'").orderBy("event_time").limit(20))

## Step 5. Gold 層 — 時間窓ごとのエラー集計と急増検知

10分単位のウィンドウで ERROR 件数を集計します。平常時はほぼ 0〜1 件ですが、
インシデント時間帯だけ件数が跳ね上がるはずです。


In [ ]:
# 10分ウィンドウ × エラー件数 (アプリ側)
gold_app = (spark.table("silver_app")
    .filter("level = 'ERROR'")
    .groupBy(F.window("event_time", "10 minutes").alias("w"))
    .agg(F.count("*").alias("error_count"),
         F.collect_set("exception_class").alias("exception_types"),
         F.collect_set(F.when(F.col("ora_code") != "", F.col("ora_code"))).alias("related_ora"))
    .select(F.col("w.start").alias("window_start"),
            "error_count", "exception_types", "related_ora")
    .orderBy("window_start"))

gold_app.write.mode("overwrite").saveAsTable("gold_app_errors_10m")
display(gold_app.filter("error_count > 0"))

### しきい値ベースのアラート検知

「10分窓で ERROR が5件を超えたら異常」というシンプルなルールで検知します。
実務ではここを移動平均からの乖離 (例: 平常時平均 + 3σ) などに発展させます。


In [ ]:
%sql
SELECT
  window_start,
  error_count,
  exception_types,
  related_ora,
  CASE WHEN error_count > 5 THEN '🚨 ALERT' ELSE 'ok' END AS status
FROM gold_app_errors_10m
WHERE error_count > 5
ORDER BY window_start;

## Step 6. DB ログ × アプリログの相関分析

ここがこのハンズオンの肝です。同じ10分窓で **Oracle 側エラー** と **アプリ側エラー** を
横並びにし、`ORA-` コードで突き合わせます。アプリの障害が DB のデッドロック /
接続枯渇に起因していることが1枚のビューで見えます。


In [ ]:
%sql
WITH ora AS (
  SELECT window(event_time, '10 minutes').start AS window_start,
         count(*) AS oracle_errors,
         collect_set(ora_code) AS oracle_ora_codes
  FROM silver_oracle WHERE severity = 'ERROR'
  GROUP BY 1
),
app AS (
  SELECT window(event_time, '10 minutes').start AS window_start,
         count(*) AS app_errors,
         collect_set(CASE WHEN ora_code <> '' THEN ora_code END) AS app_seen_ora_codes
  FROM silver_app WHERE level = 'ERROR'
  GROUP BY 1
)
SELECT
  coalesce(ora.window_start, app.window_start) AS window_start,
  coalesce(oracle_errors, 0) AS oracle_errors,
  coalesce(app_errors, 0)    AS app_errors,
  oracle_ora_codes,
  app_seen_ora_codes
FROM ora FULL OUTER JOIN app USING (window_start)
WHERE coalesce(oracle_errors,0) > 0 OR coalesce(app_errors,0) > 0
ORDER BY window_start;

**読み取り方:** インシデント窓 (10:00 台) では `oracle_errors` と `app_errors` が
同時に跳ね上がり、Oracle 側の `ORA-00060 / ORA-12516` が、アプリ側で観測された
`app_seen_ora_codes` と一致します → **DB のデッドロック＋接続枯渇がアプリ障害の根本原因**、
と切り分けられます。一方で 02:30 の `NullPointerException` (バッチ) は DB と無関係な単発障害、
と区別できます。


## Step 7. 発展課題 (Free Edition で試せる範囲)

- **トレース内容の深掘り:** `silver_trace` から `current_sql` を集計し、デッドロックに
  関与する SQL/テーブルの傾向を見る (今回は orders / order_items / inventory が頻出)。
- **trc を Gold に接続:** デッドロック窓ごとに、関与した SQL・セッション・モジュールを
  まとめた Gold テーブルを作り、根本原因の調査材料にする。
- **ダッシュボード化:** `gold_app_errors_10m` を SQL エディタでクエリし、折れ線グラフにして
  Databricks Dashboard に固定。エラー件数の時系列を可視化する。
- **AI/BI Genie:** Genie スペースに `silver_app` / `silver_oracle` / `silver_trace` を登録し、
  「10時台のデッドロックに関与した SQL は?」と日本語で質問してみる。
- **Lakeflow パイプライン化:** Bronze→Silver→Gold を宣言的パイプラインに移し、
  ファイルが Volume に届いたら自動で増分処理されるようにする。
- **検知ロジックの高度化:** 固定しきい値 (>5) を、過去N窓の移動平均 + 3σ に置き換える。

> 片付け: 不要になったら `DROP CATALOG loghandson CASCADE;` で一括削除できます
> (Free Edition のクォータ節約)。
